<a href="https://colab.research.google.com/github/karin-kaito/202604-llm/blob/main/FT_8_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.1.7 LoRA ファインチューニング

In [ ]:
!pip install -qqq trl==0.18.2

In [ ]:
%%python
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

# 事前学習済みモデルをロードします。
# 今回はQwenの0.5Bパラメータモデルを使用します。
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B"
)

# LoRA (Low-Rank Adaptation) の設定を定義します。
# LoRAは、事前学習済みモデルのすべての重みをファインチューニングするのではなく、
# 一部の層に小さな「アダプター」を導入し、そのアダプターのみを学習させることで、
# 計算コストとメモリ使用量を削減しつつ、モデルの性能を向上させる手法です。
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # タスクタイプを因果的言語モデリングに設定
    r=8, # LoRAのランク。アダプターの低ランク近似の次元数を制御します。
         # rが大きいほど表現力が増しますが、パラメータ数も増加します。
    target_modules=[
        "q_proj", # 注意機構のクエリ射影層
        "k_proj", # 注意機構のキー射影層
        "v_proj", # 注意機構のバリュー射影層
        "o_proj", # 注意機構の出力射影層
        "gate_proj", # MLPのゲート層 (Qwenでは通常)
        "up_proj", # MLPのアップ射影層
        "down_proj" # MLPのダウン射影層
    ], # LoRAアダプターを適用する対象のモジュール（層）を指定します。
       # これらの層に小さな重み行列が追加され、それらが学習されます。
    layers_to_transform=None, # 特定の層のみを変換する場合に指定しますが、今回はすべての該当層に適用します。
    lora_alpha=32, # LoRAのスケール係数。学習された重みをスケーリングします。
                   # rとlora_alphaはLoRAアダプターの重みの初期化とスケーリングに影響します。
    lora_dropout=0.1 # LoRAアダプターのドロップアウト率。過学習を防ぐために使用します。
)

# ファインチューニング用のデータセットを辞書から作成します。
# このデータセットは、プロンプトとその期待される完了（応答）のペアを含みます。
ds = Dataset.from_dict({
    "prompt": [
        "日本語で「Hello, world!」と言ってください。",
        "2 と 3 を掛け算してください。",
    ],
    "completion": [
        "こんにちは、世界！",
        "6",
    ],
})

# SFTTrainer (Supervised Fine-Tuning Trainer) のトレーニング引数を設定します。
# これはHugging FaceのTrainerクラスを基盤としており、SFTに特化しています。
training_args = SFTConfig(
    output_dir="./ckpt-sft-lora", # 学習済みモデルとチェックポイントの保存先ディレクトリ
    num_train_epochs=1, # 全データセットに対する学習エポック数
    fp16=True, # 混合精度学習（Float16）を有効にします。メモリ使用量を削減し、学習速度を向上させます。
    report_to="none", # レポート機能を無効にします（wandbなどの統合をしない場合）。
    logging_steps=1, # ログを出力するステップ間隔
)

# SFTTrainerを初期化します。
# モデル、トレーニングデータセット、トレーニング引数、LoRA設定を渡します。
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=training_args,
    peft_config=peft_config,
)

# モデルの学習を開始します。
trainer.train()

# 学習したLoRAアダプタのみを保存します。
# これにより、ベースモデル全体を保存するよりもはるかに小さなファイルサイズでファインチューニングの結果を保持できます。
trainer.save_model("./ckpt-sft-lora/adapters")

### 学習前後の性能比較

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# トークナイザーのロード
# これはテキストをモデルが理解できるトークンIDに変換するために必要です。
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
tokenizer.pad_token = tokenizer.eos_token # パディングトークンをEOSトークンに設定。これはバッチ処理でシーケンス長を揃えるために使われます。

In [ ]:
# ベースモデルのロード（量子化なし、必要に応じて設定）
# 前回、BitsAndBytesConfigが使われていないことを確認したため、ここでは通常のロードを行います。
# LoRAアダプターをマージするためには、ベースモデルと同じ精度でロードする必要があります。
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    torch_dtype=torch.bfloat16, # 学習時にfp16が使われたのでbfloat16でロード
) # モデルをbfloat16のデータ型でロードします。これによりメモリ使用量が抑えられ、
  # fp16での学習と互換性が高まります。
base_model.eval() # 評価モードに設定。これにより、Dropout層などが無効になり、推論結果が安定します。

In [ ]:
# LoRAアダプターをロードし、ベースモデルとマージしてファインチューニング済みモデルを作成
# PeftModel.from_pretrained() を使用して、学習済みのLoRAアダプターをベースモデルに適用します。
finetuned_model = PeftModel.from_pretrained(base_model, "./ckpt-sft-lora/adapters")
# アダプターをベースモデルにマージします。
# これにより、LoRAアダプターの重みがベースモデルの対応する層に統合され、
# 以降の推論でPeftModelのラッパーを使用せずに直接モデルを使用できるようになります。
finetuned_model = finetuned_model.merge_and_unload()
finetuned_model.eval() # 評価モードに設定

In [ ]:
def generate_response(model, tokenizer, prompt):
    # プロンプトをトークン化し、モデルの入力形式に変換します。
    # `return_tensors="pt"`でPyTorchテンソル形式に、`padding=True`でバッチ内のシーケンス長を揃え、
    # `truncation=True`でモデルの最大入力長に合わせて切り詰めます。
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad(): # 推論時には勾配計算を無効にします。これによりメモリ使用量を減らし、計算を高速化します。
        # モデルのgenerateメソッドを使用してテキストを生成します。
        # `max_new_tokens`で生成する最大トークン数を指定。
        # `num_return_sequences=1`で1つの生成結果を返します。
        # `do_sample=False`で決定論的な生成（貪欲法）を行います。
        # `temperature=0.1`はサンプリング時の多様性を制御しますが、do_sample=Falseなのでここでは効果がありません。
        # `pad_token_id`はパディングトークンのIDを設定します。
        outputs = model.generate(**inputs, max_new_tokens=50, num_return_sequences=1, do_sample=False, temperature=0.1, pad_token_id=tokenizer.eos_token_id)
    # 生成された出力からプロンプト部分を除外してデコードします。
    # `skip_special_tokens=True`で特殊トークン（EOS、BOSなど）を結果から除外します。
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

In [ ]:
# 評価プロンプトの準備（トレーニングデータセットから取得）
eval_prompts = [
    "日本語で「Hello, world!」と言ってください。",
    "2 と 3 を掛け算してください。",
]
expected_completions = [
    "こんにちは、世界！",
    "6",
]

print("--- モデル出力比較 ---")
for i, prompt in enumerate(eval_prompts):
    expected = expected_completions[i]

    # ベースモデルでの生成
    base_model_response = generate_response(base_model, tokenizer, prompt)

    # ファインチューニング済みモデルでの生成
    finetuned_model_response = generate_response(finetuned_model, tokenizer, prompt)

    print(f"\nプロンプト: {prompt}")
    print(f"期待される応答: {expected}")
    print(f"ベースモデルの応答: {base_model_response}")
    print(f"ファインチューニングモデルの応答: {finetuned_model_response}")
    print("---------------------")

### ハイパーパラメータ調整の例

前回の学習では期待する効果が得られなかったため、学習率、バッチサイズ、およびエポック数を調整して、再度ファインチューニングを試みる例です。

*   `learning_rate`: 学習の進み具合を制御します。小さすぎると学習が遅く、大きすぎると不安定になる可能性があります。
*   `per_device_train_batch_size`: 一度に処理されるデータの数を指定します。大きいほどGPUメモリを多く消費しますが、学習の安定性や速度が向上することがあります。
*   `num_train_epochs`: モデルがデータセット全体を何回学習するかを決定します。今回は `1` から `3` に増やします。

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

# モデルとトークナイザーのロード (前回のコードから再利用)
# 事前学習済みモデルをロードします。今回はQwenの0.5Bパラメータモデルを使用します。
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    torch_dtype=torch.bfloat16 # fp16で学習したのでbfloat16でロード
) # モデルをbfloat16のデータ型でロードします。これによりメモリ使用量が抑えられ、
  # fp16での学習と互換性が高まります。
# トークナイザーもロードし、パディングトークンを設定
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
tokenizer.pad_token = tokenizer.eos_token

# LoRA 設定 (前回のコードから再利用)
# LoRA (Low-Rank Adaptation) の設定を定義します。
# LoRAは、事前学習済みモデルのすべての重みをファインチューニングするのではなく、
# 一部の層に小さな「アダプター」を導入し、そのアダプターのみを学習させることで、
# 計算コストとメモリ使用量を削減しつつ、モデルの性能を向上させる手法です。
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # タスクタイプを因果的言語モデリングに設定
    r=8, # LoRAのランク。アダプターの低ランク近似の次元数を制御します。
         # rが大きいほど表現力が増しますが、パラメータ数も増加します。
    target_modules=[
        "q_proj", # 注意機構のクエリ射影層
        "k_proj", # 注意機構のキー射影層
        "v_proj", # 注意機構のバリュー射影層
        "o_proj", # 注意機構の出力射影層
        "gate_proj", # MLPのゲート層 (Qwenでは通常)
        "up_proj", # MLPのアップ射影層
        "down_proj" # MLPのダウン射影層
    ], # LoRAアダプターを適用する対象のモジュール（層）を指定します。
       # これらの層に小さな重み行列が追加され、それらが学習されます。
    layers_to_transform=None, # 特定の層のみを変換する場合に指定しますが、今回はすべての該当層に適用します。
    lora_alpha=32, # LoRAのスケール係数。学習された重みをスケーリングします。
                   # rとlora_alphaはLoRAアダプターの重みの初期化とスケーリングに影響します。
    lora_dropout=0.1 # LoRAアダプターのドロップアウト率。過学習を防ぐために使用します。
)

# データセット (前回のコードから再利用)
# ファインチューニング用のデータセットを辞書から作成します。
# このデータセットは、プロンプトとその期待される完了（応答）のペアを含みます。
# ds = Dataset.from_dict({
#     "prompt": [
#         "日本語で「Hello, world!」と言ってください。",
#         "2 と 3 を掛け算してください。",
#     ],
#     "completion": [
#         "こんにちは、世界！",
#         "6",
#     ],
# })

# ハイパーパラメータを調整した新しいSFTConfig
# ここで learning_rate, per_device_train_batch_size, num_train_epochs を調整します
# SFTTrainer (Supervised Fine-Tuning Trainer) のトレーニング引数を設定します。
# これはHugging FaceのTrainerクラスを基盤としており、SFTに特化しています。
training_args_adjusted = SFTConfig(
    output_dir="./ckpt-sft-lora-expanded", # 出力ディレクトリを変更
    num_train_epochs=3, # エポック数を1から3に増加
    fp16=True, # 混合精度学習（Float16）を有効にします。メモリ使用量を削減し、学習速度を向上させます。
    report_to="none", # レポート機能を無効にします（wandbなどの統合をしない場合）。
    logging_steps=1, # ログを出力するステップ間隔
    learning_rate=2e-5, # 学習率を調整。学習の進み具合を制御します。
    per_device_train_batch_size=1, # バッチサイズを調整 (例: 1)。一度に処理されるデータの数を指定します。
    # gradient_accumulation_steps=4, # バッチサイズが小さい場合、勾配累積で実質的なバッチサイズを増やす
)

# SFTTrainerを初期化します。
# モデル、トレーニングデータセット、トレーニング引数、LoRA設定を渡します。
trainer_expanded = SFTTrainer(
    model=model,
    train_dataset=ds_expanded, # ここをds_expandedに変更
    args=training_args_adjusted,
    peft_config=peft_config
    # tokenizer=tokenizer # トークナイザーを明示的に渡す (このtrlのバージョンでは不要)
)

# モデルの学習を開始します。
trainer_expanded.train()

# 調整後モデルのLoRAアダプタを保存
# 学習したLoRAアダプタのみを保存します。
# これにより、ベースモデル全体を保存するよりもはるかに小さなファイルサイズでファインチューニングの結果を保持できます。
trainer_expanded.save_model("./ckpt-sft-lora-expanded/adapters")

### 調整後のモデルで再度性能比較

ハイパーパラメータを調整したモデルで再度生成を行い、ベースモデルや前回のファインチューニングモデルと比較します。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# トークナイザーのロード（必要であれば再ロード）
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
# tokenizer.pad_token = tokenizer.eos_token

# ベースモデルのロード (再度使用するため、ここでもロードしておきます)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    torch_dtype=torch.bfloat16 # fp16で学習したのでbfloat16でロード
)
base_model.eval() # 評価モードに設定

# 調整後のLoRAアダプターをロードし、ベースモデルとマージ
# PeftModel.from_pretrained() を使用して、学習済みのLoRAアダプターをベースモデルに適用します。
finetuned_model_expanded = PeftModel.from_pretrained(base_model, "./ckpt-sft-lora-expanded/adapters")
# アダプターをベースモデルにマージします。
# これにより、LoRAアダプターの重みがベースモデルの対応する層に統合され、
# 以降の推論でPeftModelのラッパーを使用せずに直接モデルを使用できるようになります。
finetuned_model_expanded = finetuned_model_expanded.merge_and_unload()
finetuned_model_expanded.eval() # 評価モードに設定

# 評価プロンプトの準備（トレーニングデータセットから取得）
eval_prompts_expanded = [
    "日本語で「Hello, world!」と言ってください。",
    "2 と 3 を掛け算してください。",
    "Pythonで「Hello, world!」を出力するコードを書いてください。",
    "「ありがとう」を英語で何と言いますか？",
    "5 に 7 を足してください。",
    "日本の首都はどこですか？",
    "今日の天気は？",
    "明日、何をすべきですか？",
    "10 から 4 を引いてください。",
    "円周率の最初の3桁を教えてください。"
]
expected_completions_expanded = [
    "こんにちは、世界！",
    "6",
    "print('Hello, world!')",
    "Thank you.",
    "12",
    "東京です。",
    "今日の天気は晴れです。",
    "明日は買い物に行きましょう。",
    "6",
    "3.14"
]

print("--- 拡充後のモデル出力比較 ---")
for i, prompt in enumerate(eval_prompts_expanded):
    expected = expected_completions_expanded[i]

    # ベースモデルでの生成 (再利用)
    base_model_response = generate_response(base_model, tokenizer, prompt)

    # 調整後のファインチューニングモデルでの生成
    finetuned_model_expanded_response = generate_response(finetuned_model_expanded, tokenizer, prompt)

    print(f"\nプロンプト: {prompt}")
    print(f"期待される応答: {expected}")
    print(f"ベースモデルの応答: {base_model_response}")
    print(f"拡充されたファインチューニングモデルの応答: {finetuned_model_expanded_response}")
    print("---------------------")

### ファインチューニング済みモデルの構造とLoRAパラメータ設定

In [ ]:
# ファインチューニング済みモデルの構造を表示
# これには、モデルのレイヤー構成などが含まれます。
print("ファインチューニング済みモデル (finetuned_model_expanded) の構造:")
print(finetuned_model_expanded)

In [ ]:
# LoRAのパラメータ設定を表示
# LoRAConfigオブジェクトには、r, lora_alpha, target_modulesなどの設定が含まれています。
print("\nLoRAのパラメータ設定 (peft_config):")
print(peft_config)

### データセット拡充前後の性能比較

In [ ]:
# 評価プロンプトの準備 (拡充されたデータセットで使用したものと同じものを使用)
# eval_prompts_expanded, expected_completions_expanded はすでに定義されています

print("--- データセット拡充前後のモデル出力比較 ---")
for i, prompt in enumerate(eval_prompts_expanded):
    expected = expected_completions_expanded[i]

    # ベースモデルでの生成
    base_model_response = generate_response(base_model, tokenizer, prompt)

    # 最初のファインチューニングモデルでの生成 (データセット拡充前)
    # finetuned_model は最初の学習で保存されたモデルをロードしています
    finetuned_model_prev_response = generate_response(finetuned_model, tokenizer, prompt)

    # 拡充されたデータセットで学習したファインチューニングモデルでの生成
    finetuned_model_expanded_response = generate_response(finetuned_model_expanded, tokenizer, prompt)

    print(f"\nプロンプト: {prompt}")
    print(f"期待される応答: {expected}")
    print(f"ベースモデルの応答: {base_model_response}")
    print(f"初期ファインチューニングモデルの応答: {finetuned_model_prev_response}")
    print(f"拡充後ファインチューニングモデルの応答: {finetuned_model_expanded_response}")
    print("---------------------")

### データセット拡充の例

モデルの性能を向上させるためには、より多くの多様な学習データが必要です。ここでは、既存のデータセットに加えて、さらに多くのプロンプトと期待される完了を追加したデータセットの例を示します。データセットを増やすことで、モデルがより一般的なパターンや、与えられたタスクに対するより正確な応答を学習できるようになります。

In [ ]:
# 拡充されたデータセットを辞書から作成します。
# これにより、モデルがより多様な入出力パターンを学習できるようになります。
ds_expanded = Dataset.from_dict({
    "prompt": [
        "日本語で「Hello, world!」と言ってください。",
        "2 と 3 を掛け算してください。",
        "Pythonで「Hello, world!」を出力するコードを書いてください。",
        "「ありがとう」を英語で何と言いますか？",
        "5 に 7 を足してください。",
        "日本の首都はどこですか？",
        "今日の天気は？",
        "明日、何をすべきですか？",
        "10 から 4 を引いてください。",
        "円周率の最初の3桁を教えてください。"
    ],
    "completion": [
        "こんにちは、世界！",
        "6",
        "print('Hello, world!')",
        "Thank you.",
        "12",
        "東京です。",
        "今日の天気は晴れです。",
        "明日は買い物に行きましょう。",
        "6",
        "3.14"
    ],
})

print(f"拡充されたデータセットのサンプル数: {len(ds_expanded)}")
print(ds_expanded[0])
print(ds_expanded[-1])

この`ds_expanded`データセットを使って、再度モデルをファインチューニングすることができます。その際には、`SFTTrainer`の`train_dataset`引数に`ds_expanded`を指定します。

```python
# 例: 拡充されたデータセットで再学習を行う場合
# trainer_expanded = SFTTrainer(
#     model=model,
#     train_dataset=ds_expanded, # ここをds_expandedに変更
#     args=training_args_adjusted, # 適切なtraining_argsを使用
#     peft_config=peft_config
# )
# trainer_expanded.train()
# trainer_expanded.save_model("./ckpt-sft-lora-expanded/adapters")
```

これにより、モデルはより多くの情報から学習し、新しいプロンプトに対しても適切な応答を生成する可能性が高まります。

In [ ]:
!ls -la ./ckpt-sft-lora

In [ ]:
!ls -la ./ckpt-sft-lora/*